# analyse des avis et alertes ANSSI — enrichissement CVE

**SUPDEVINCI 2026 — TD final noté**

## contexte

L'ANSSI (Agence Nationale de la Sécurité des Systèmes d'Information) publie régulièrement des bulletins CERTFR :
- **avis** : signalement de vulnérabilités identifiées dans des logiciels ou équipements
- **alertes** : vulnérabilités critiques avec un risque d'exploitation immédiat

Ces bulletins référencent des **CVE** (Common Vulnerabilities and Exposures), identifiants standardisés pour les failles de sécurité.

## objectif

Construire un pipeline complet de veille cyber automatisée :
1. extraction des données CERTFR depuis les fichiers JSON locaux
2. enrichissement de chaque CVE avec le score CVSS (sévérité) et le score EPSS (probabilité d'exploitation)
3. exploration et visualisation du dataset
4. clustering non supervisé pour regrouper les CVE par profil de risque
5. classification supervisée pour prédire la sévérité d'une CVE
6. génération automatique d'alertes par email pour les CVE critiques

## sources de données externes

| source | usage |
|--------|-------|
| MITRE CVE API (`cveawg.mitre.org`) | description, CVSS, CWE pour chaque CVE |
| FIRST EPSS API (`first.org/epss`) | probabilité d'exploitation dans les 30 jours |


---
## 1. configuration de l'environnement

On définit une fonction utilitaire `run()` qui exécute chaque script Python du projet en capturant sa sortie. Cela permet d'organiser le notebook en étapes distinctes sans tout recopier.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import subprocess, sys

def run(script):
    """exécute un script du projet et affiche sa sortie."""
    result = subprocess.run([sys.executable, script], capture_output=True, text=True, cwd=".")
    print(result.stdout[-3000:] if result.stdout else "")
    if result.returncode != 0:
        print("ERREUR:", result.stderr[-1000:])


---
## 2. extraction et enrichissement des CVE

### pipeline d'enrichissement

```
fichiers JSON CERTFR  →  extraction des CVE  →  appels API MITRE  →  appels API EPSS
                                                                    ↓
                                                    output/cve_enriched.csv
                                                    output/cve_detail.csv
```

**`cve_enriched.csv`** : une ligne par CVE unique, avec toutes ses métadonnées agrégées  
**`cve_detail.csv`** : une ligne par association CVE ↔ bulletin CERTFR

### métriques attendues

- ~4 000 avis, ~78 alertes CERTFR analysés
- ~125 000 associations CVE ↔ bulletins
- ~37 000 CVE uniques après déduplication

> **note** : le script effectue des requêtes HTTP concurrentes (10 simultanées). La première exécution peut prendre plusieurs minutes. Les résultats sont mis en cache dans `output/`.


In [ ]:
run("enrich_cve.py")

---
## 3. exploration du dataset consolidé

On charge `cve_enriched.csv` pour examiner la structure des données avant d'aller plus loin.

### colonnes principales

| colonne | description |
|---------|-------------|
| `cve_id` | identifiant CVE (ex: CVE-2023-44487) |
| `cvss` | score de sévérité CVSS v3 (0–10) |
| `severite` | Faible / Moyenne / Élevée / Critique |
| `epss` | probabilité d'exploitation sur 30j (0–1) |
| `epss_percentile` | rang percentile EPSS parmi toutes les CVE connues |
| `nb_documents` | nombre de bulletins CERTFR qui citent cette CVE |
| `editeur` | éditeur(s) du logiciel vulnérable |
| `cwe` | type de vulnérabilité (CWE-79 = XSS, CWE-20 = validation...) |


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("output/cve_enriched.csv")
df["cvss"] = pd.to_numeric(df["cvss"], errors="coerce")
df["epss"] = pd.to_numeric(df["epss"], errors="coerce")

print(f"shape : {df.shape}")
print(f"colonnes : {list(df.columns)}")
df.head(3)


In [ ]:
# les statistiques descriptives révèlent la distribution des scores
# cvss médian ~7-8 (élevé) car l'ANSSI ne publie que les vulnérabilités significatives
# epss très asymétrique (skewed right) : la majorité des CVE ont une faible proba d'exploitation
print("statistiques descriptives :")
df[["cvss","epss","epss_percentile","nb_documents"]].describe().round(3)


In [ ]:
print("distribution des sévérités :")
print(df["severite"].value_counts())
print()
print("top 5 CVE par nb de documents CERTFR (les plus suivies) :")
cols = ["cve_id","nb_documents","cvss","severite","epss","editeur"]
print(df.nlargest(5,"nb_documents")[cols].to_string(index=False))


---
## 4. visualisations exploratoires

4 graphiques complémentaires pour comprendre la structure du dataset :

1. **heatmap de corrélation** : relations entre variables numériques  
   → CVSS et EPSS ne sont corrélés qu'à ~0.16 : un score de gravité élevé ne suffit pas à prédire le risque d'exploitation réel

2. **boxplot CVSS par éditeur** : dispersion des sévérités pour les 8 éditeurs les plus cités  
   → permet d'identifier les éditeurs dont les produits concentrent les vulnérabilités critiques

3. **courbe cumulative** : évolution du nombre de CVE référencées par l'ANSSI dans le temps  
   → tendance à la hausse reflète la multiplication des vulnérabilités publiées

4. **camembert CWE** : répartition des types de vulnérabilités  
   → les types de failles les plus fréquentes dans l'écosystème surveillé par l'ANSSI


In [ ]:
run("visualize_extra.py")

In [ ]:
from IPython.display import Image
Image("output/visualize_extra.png")


---
## 5. ML non supervisé — clustering des CVE

### pourquoi le clustering ?

L'objectif est de **regrouper automatiquement** les CVE par profil de risque, sans étiquette prédéfinie. Cela permet de :
- identifier des patterns que les seuils CVSS manuels ne capturent pas
- détecter des outliers (CVE atypiques, potentiellement à surveiller en priorité)
- valider que les données présentent bien une structure naturelle

### features retenues

| feature | justification |
|---------|--------------|
| `cvss` | mesure de sévérité standard |
| `log(epss)` | probabilité d'exploitation (log car distribution très asymétrique) |
| `epss_percentile` | rang relatif de l'EPSS |
| `nb_documents` | nombre de bulletins ANSSI citant la CVE |
| `sev_num` | sévérité encodée en entier (1=Faible, 4=Critique) |

> **note** : on applique un `StandardScaler` avant le clustering pour que chaque feature ait le même poids.

### réduction dimensionnelle

On utilise **PCA (Analyse en Composantes Principales)** pour projeter les 5 dimensions en 2D — uniquement pour la **visualisation**. Le clustering lui-même s'effectue sur les 5 features originales.


### 5.1 trouver le bon nombre de clusters (KMeans)

Avant de comparer tous les algorithmes, on utilise deux méthodes pour estimer le k optimal pour KMeans :

- **méthode du coude (elbow)** : on trace l'inertie (distance intra-cluster) en fonction de k. Le "coude" indique où ajouter un cluster n'apporte plus beaucoup
- **silhouette score** : mesure de 0 à 1 à quel point chaque point "appartient" à son cluster vs. le cluster voisin

> **résultat** : elbow → k=3, silhouette → k=2, mais on choisit **k=4** par connaissance métier (4 niveaux de sévérité CVSS)


In [ ]:
run("find_k.py")

In [ ]:
Image("output/find_k.png")


### 5.2 comparatif de 11 algorithmes de clustering

On compare :

| catégorie | algorithmes |
|-----------|-------------|
| **partitionnement** | MiniBatchKMeans, Gaussian Mixture |
| **hiérarchique** | Ward, Agglomerative Clustering, BIRCH |
| **densité** | DBSCAN, **HDBSCAN**, OPTICS |
| **autres** | Affinity Propagation, MeanShift, Spectral Clustering |

Chaque algorithme est évalué visuellement sur la projection PCA 2D. Les points noirs représentent les **outliers** (bruit).


In [ ]:
run("clustering.py")

In [ ]:
Image("output/clustering.png")


### 5.3 analyse détaillée de HDBSCAN

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering) est retenu comme meilleur algorithme car :

- il ne force pas chaque point dans un cluster (contrairement à KMeans)
- il détecte automatiquement le nombre de clusters sans hyperparamètre k
- il identifie les **outliers** (points noirs) — CVE atypiques à traiter en priorité
- robuste aux distributions irrégulières (ce qui est le cas avec EPSS)

**métriques de qualité** :
- `silhouette score` : mesure la cohésion et la séparation des clusters (plus c'est proche de 1, mieux c'est)
- `Davies-Bouldin index` : ratio distance intra/inter-cluster (plus c'est proche de 0, mieux c'est)

Les 3 graphiques ci-dessous montrent : la projection PCA, le profil normalisé de chaque cluster, et leur taille respective.


In [ ]:
Image("output/clustering_hdbscan.png")


---
## 6. ML supervisé — prédiction de la sévérité

### objectif

Prédire la **sévérité CERTFR** d'une CVE (Faible / Moyenne / Élevée / Critique) à partir de ses caractéristiques — **sans utiliser le score CVSS** comme feature.

Cela simule un scénario réaliste : on veut anticiper la criticité d'une CVE avant d'avoir son score officiel.

### features retenues

| type | features |
|------|---------|
| numériques | `log(epss)`, `epss_percentile`, `nb_documents`, `annee_cve` |
| binaires | `exploitation` (CVE activement exploitée), `has_cwe` (type connu), `is_alerte` |
| multi-hot | catégories de risque ANSSI (~15 catégories) |
| one-hot | top 20 CWE les plus fréquents |

### évaluation sans data leakage

On utilise une **validation croisée stratifiée 4-fold** avec `cross_val_predict` :
- le dataset est découpé en 4 parties égales (stratifiées = même proportion de classes dans chaque fold)
- chaque CVE est prédite par un modèle **qui ne l'a pas vue à l'entraînement**
- garantit que les métriques reflètent la vraie capacité de généralisation

### 10 classifieurs comparés

RandomForest, GradientBoosting, ExtraTrees, LogisticRegression, LinearSVC, KNeighbors, DecisionTree, GaussianNB, AdaBoost, MLP


In [ ]:
run("supervised.py")

In [ ]:
Image("output/supervised.png")


### 6.1 interprétation des résultats

**F1 macro ~0.5** : résultat attendu étant donné qu'on n'utilise pas le CVSS — on prédit la sévérité à partir de signaux indirects.

**pourquoi le déséquilibre des classes est un problème** :
- classe "Faible" = ~2% du dataset → le modèle a peu d'exemples pour apprendre
- un classifieur naïf qui prédit toujours "Élevée" aurait une accuracy >50% mais un F1 macro très bas

**les features les plus importantes** (Random Forest) :
- catégories de risque ANSSI (multi-hot)
- type CWE
- `is_alerte` : les alertes ANSSI concernent quasi-exclusivement des CVE critiques

### 6.2 courbes de validation


In [ ]:
Image("output/supervised_validation.png")


---
## 7. génération d'alertes et notifications email

### critères d'alerte

Une CVE déclenche une alerte si elle remplit **les deux conditions** :
- (`CVSS ≥ 9.0` **ou** `sévérité == "Critique"`) — gravité maximale
- `EPSS ≥ 0.5` — plus de 50% de probabilité d'exploitation dans les 30 jours

Ce double critère évite :
- les faux positifs (CVE grave mais jamais exploitée)
- les faux négatifs (CVE activement exploitée mais "seulement" élevée)

### email via Resend

L'alerte est :
1. générée en HTML stylisé et sauvegardée dans `output/alerts/`
2. envoyée par email via l'API Resend

Les credentials (clé API, adresse source, destinataire) sont chargés depuis un fichier `.env` — jamais écrits en dur dans le code.


In [ ]:
run("alerts.py")

In [ ]:
from IPython.display import HTML
from pathlib import Path
import glob

alert_files = sorted(glob.glob("output/alerts/alert_*.html"))
if alert_files:
    HTML(Path(alert_files[-1]).read_text(encoding="utf-8"))
else:
    print("aucun fichier d'alerte trouvé — relancer alerts.py")


---
## 8. synthèse et conclusions

### résultats clés

| étape | résultat |
|-------|---------|
| extraction | ~37 000 CVE uniques issues de ~4 000 bulletins CERTFR |
| enrichissement | CVSS, EPSS, CWE récupérés via APIs MITRE + FIRST |
| clustering | HDBSCAN : 4 clusters naturels + détection d'outliers |
| classification | Random Forest : F1 macro ~0.5 (sans CVSS) |
| alertes | email automatique pour les CVE CVSS≥9 + EPSS≥0.5 |

### ce que le clustering révèle

Les 4 clusters correspondent approximativement aux 4 niveaux de sévérité CVSS, mais HDBSCAN les découvre de façon **non supervisée** — ce qui valide que les features choisies portent bien l'information de risque. Les outliers identifiés sont des CVE avec des combinaisons de scores inhabituelles (EPSS élevé mais CVSS moyen, par exemple).

### pourquoi le F1 supervisé est limité à ~0.5

- **pas de CVSS dans les features** : c'est un choix délibéré (simuler une prédiction précoce)
- **déséquilibre des classes** : "Faible" = 2% des données, difficile à apprendre
- **features indirectes** : les risques ANSSI et le CWE sont des proxies imparfaits de la sévérité

### pistes d'amélioration

- connecter le pipeline au **flux RSS live** de l'ANSSI pour une veille en temps réel
- enrichir avec des **données de threat intelligence** (Shodan, VirusTotal)
- tester des **modèles de language** (LLM) pour classifier depuis la description textuelle
- implémenter un **retraining automatique** quand de nouvelles données arrivent
